# Gold — modelo dimensional (etapa 4.1)

Fonte de pipeline do **novo pipeline `olist.gold`**. Lê as tabelas da silver (que o pipeline `olist.silver` mantém) e declara **materialized views** com o modelo estrela:

| Tabela | Granularidade | O que guarda |
|---|---|---|
| `dim_date` | 1 linha por dia | calendário, do primeiro dia de compra até 30 dias depois da última entrega estimada |
| `dim_product` | 1 linha por produto | categoria em inglês (`desconhecida` quando não há) |
| `dim_customer` | 1 linha por **versão** do cadastro | `customer_sk` (chave da versão), validade, `cadastro_excluido_hoje` |
| `dim_seller` | 1 linha por **versão** do cadastro | `seller_sk`, validade, `cadastro_excluido_hoje` |
| `fact_orders` | 1 linha por pedido | status atual, datas, flags, GMV, lead time, nota mais recente, versão do cliente na compra |
| `fact_order_items` | 1 linha por item | preço, frete, produto, vendedor, versão do vendedor na compra |
| `fact_reviews` | 1 linha por (review, pedido) | `eh_mais_recente`, `pedido_existe`, sem o texto do comentário |
| `gold_seller_mensal` | 1 linha por vendedor e mês | GMV e posição do vendedor no mês (etapa 4.2, base da concentração de vendas) |

Decisões (`docs/design/fase-4-gold-metricas.md`): G1 materialized views em pipeline próprio · G2 status atual = maior `event_sequence`, entrega = `event_ts` do evento `delivered` · G3 fatos de item, pedido e review · G4 chave substituta por versão, junção por intervalo feita **uma vez** aqui · G8 `CLUSTER BY` nos fatos.

**Este arquivo é fonte de pipeline, não notebook comum** (roteiro na última célula).

```
silver.orders · order_items · order_status_events · order_reviews · products · category_translation
silver.customers_historico · sellers_historico (SCD2)
      └─► dim_date · dim_product · dim_customer · dim_seller
              └─► fact_orders ──► fact_order_items
silver.order_reviews ───────────► fact_reviews
```

## Configuração

Importa as bibliotecas, lê o catálogo da configuração do pipeline e declara dois atalhos de leitura (`silver` e `gold`). Os nomes das tabelas são sempre completos (`olist.gold.<tabela>`), assim o mesmo arquivo funciona num pipeline próprio ou, no plano reserva, dentro do `olist.silver`.

In [0]:
# O QUE FAZ: importa o módulo do pipeline, Window e as funções de coluna, lê o catálogo da
#   configuração do pipeline e declara duas funções de leitura. Nenhum dado é lido aqui.
#
# IMPORTAÇÕES:
# - from pyspark import pipelines as dp: módulo das Spark Declarative Pipelines (materialized_view).
#   Só existe dentro de um pipeline.
# - pyspark.sql.Window: define a "janela" (partição + ordem) das funções como row_number.
# - pyspark.sql.functions as F: funções nativas de coluna.
#
# FUNÇÕES DECLARADAS:
# - silver(nome): lê olist.silver.<nome> como DataFrame (leitura em lote).
# - gold(nome): lê olist.gold.<nome>. Ler uma tabela do próprio pipeline é o que cria a dependência:
#   o pipeline calcula fact_orders antes de fact_order_items porque este lê aquele.
#
# FUNÇÕES NATIVAS:
# - spark.conf.get(chave, padrão): lê um valor da configuração do pipeline; sem a chave, devolve "olist".
# - spark.read.table(nome): lê uma tabela do Unity Catalog como DataFrame.
from pyspark import pipelines as dp
from pyspark.sql import Window
from pyspark.sql import functions as F

CATALOG = spark.conf.get("olist.catalogo", "olist")


def silver(nome: str):
    return spark.read.table(f"{CATALOG}.silver.{nome}")


def gold(nome: str):
    return spark.read.table(f"{CATALOG}.gold.{nome}")

## `dim_date` e `dim_product`

**`dim_date`** é gerada a partir dos próprios dados: do primeiro dia de compra até 30 dias depois da maior data estimada de entrega. Sem data fixa no código, e nada depende de "hoje" (a simulação vai até 2028).

**`dim_product`** junta o produto à tradução da categoria. A coluna `categoria` segue esta ordem: tradução em inglês → nome original em português (existem 2 categorias sem tradução, 23 itens) → `desconhecida` (623 produtos sem categoria). `categoria_origem` diz qual dos três casos ocorreu, para ninguém confundir "sem categoria" com "sem tradução".

In [0]:
# O QUE FAZ: declara duas materialized views: o calendário (dim_date) e os produtos com a
#   categoria traduzida (dim_product).
#
# FUNÇÕES DECLARADAS:
# - dim_date(): uma linha por dia entre o primeiro dia de compra e a maior entrega estimada + 30 dias.
# - dim_product(): produtos com categoria em inglês, original e a origem do valor.
#
# FUNÇÕES NATIVAS:
# - @dp.materialized_view(name, comment): declara que o resultado da função é uma materialized view
#   gravada em <name>. O pipeline cuida de criar, atualizar (incremental quando possível) e ordenar.
# - DataFrame.agg(...): uma linha com mínimo e máximo; F.min / F.max, F.to_date (timestamp → data),
#   F.date_add(data, n) soma n dias.
# - F.sequence(início, fim, passo): lista de datas de início a fim; o passo de data é um intervalo
#   (F.expr("interval 1 day")). F.explode(lista): uma linha por elemento.
# - F.date_format(data, padrão): data como texto ("yyyyMMdd", "yyyy-MM", "EEEE" = dia da semana).
#   .cast("int"): troca o tipo. F.year/F.quarter/F.month/F.dayofmonth/F.dayofweek: partes da data
#   (dayofweek: 1 = domingo ... 7 = sábado).
# - F.coalesce(a, b, c): primeiro valor não nulo. F.when(cond, valor).otherwise(valor): condicional.
# - DataFrame.join(outro, on, "left"): mantém todas as linhas da esquerda, mesmo sem correspondência.
#   .alias("p"): apelido, para separar colunas de mesmo nome (p.product_category_name).
@dp.materialized_view(
    name=f"{CATALOG}.gold.dim_date",
    comment="Calendário: uma linha por dia, da primeira compra até 30 dias depois da última entrega estimada.",
)
def dim_date():
    limites = silver("orders").agg(
        F.min(F.to_date("order_purchase_timestamp")).alias("inicio"),
        F.date_add(F.max(F.to_date("order_estimated_delivery_date")), 30).alias("fim"),
    )
    dias = limites.select(
        F.explode(F.sequence("inicio", "fim", F.expr("interval 1 day"))).alias("data")
    )
    return dias.select(
        F.date_format("data", "yyyyMMdd").cast("int").alias("date_key"),
        "data",
        F.year("data").alias("ano"),
        F.quarter("data").alias("trimestre"),
        F.month("data").alias("mes"),
        F.date_format("data", "yyyy-MM").alias("ano_mes"),
        F.dayofmonth("data").alias("dia"),
        F.dayofweek("data").alias("dia_da_semana"),
        F.date_format("data", "EEEE").alias("nome_dia_semana"),
        F.dayofweek("data").isin(1, 7).alias("fim_de_semana"),
    )


@dp.materialized_view(
    name=f"{CATALOG}.gold.dim_product",
    comment="Produtos com a categoria em inglês. categoria_origem: traduzida, original_pt (sem tradução) ou desconhecida.",
)
def dim_product():
    p = silver("products").alias("p")
    t = silver("category_translation").alias("t")
    j = p.join(t, F.col("p.product_category_name") == F.col("t.product_category_name"), "left")
    return j.select(
        F.col("p.product_id").alias("product_id"),
        F.coalesce(
            F.col("t.product_category_name_english"),
            F.col("p.product_category_name"),
            F.lit("desconhecida"),
        ).alias("categoria"),
        F.col("p.product_category_name").alias("categoria_pt"),
        F.when(F.col("t.product_category_name_english").isNotNull(), "traduzida")
        .when(F.col("p.product_category_name").isNotNull(), "original_pt")
        .otherwise("desconhecida")
        .alias("categoria_origem"),
        F.col("p.product_weight_g").alias("peso_g"),
        F.col("p.product_length_cm").alias("comprimento_cm"),
        F.col("p.product_height_cm").alias("altura_cm"),
        F.col("p.product_width_cm").alias("largura_cm"),
        F.col("p.product_photos_qty").alias("fotos_qtd"),
    )

## `dim_customer` e `dim_seller`: uma linha por versão (G4)

As duas dimensões saem das tabelas `*_historico` da Fase 3 e são criadas pela **mesma função** (`registrar_dim_versao`), como na silver. O que cada coluna nova significa:

| Coluna | O que é |
|---|---|
| `customer_sk` / `seller_sk` | chave **da versão**: hash SHA-256 de `id` + `__START_AT`. O fato guarda esta chave e o consumidor faz uma junção simples. |
| `valido_de` / `valido_ate` | os `__START_AT` e `__END_AT` do histórico; `valido_ate` nulo = versão vigente |
| `versao_vigente` | `valido_ate` é nulo |
| `cadastro_excluido_hoje` | o cadastro **não tem versão aberta** (foi excluído). Os pedidos dele continuam existindo; a Fase 6 decide o que fazer com os dados pessoais. |

In [0]:
# O QUE FAZ: para cada entidade de DIMENSOES, declara a materialized view da dimensão por versão
#   (dim_customer, dim_seller) a partir de silver.<entidade>_historico.
#
# FUNÇÕES DECLARADAS:
# - registrar_dim_versao(entidade, singular, chave, atributos): cria gold.dim_<singular>.
#
# FUNÇÕES NATIVAS:
# - F.sha2(coluna, 256): hash SHA-256 em texto hexadecimal; usado como chave substituta.
# - F.concat_ws("|", a, b): junta colunas com o separador "|" (evita que "ab"+"c" e "a"+"bc" coincidam).
# - .cast("string"): o __START_AT vira texto antes de entrar no hash.
# - F.col("__END_AT").isNull(): verdadeiro quando a versão está aberta (vigente).
# - Window.partitionBy(chave): janela com todas as versões do mesmo cadastro.
# - F.max(...).over(janela): máximo dentro da janela, sem reduzir o número de linhas. Aqui, "alguma
#   versão deste cadastro está aberta?" (1 ou 0); se o máximo for 0, o cadastro foi excluído.
# - [*lista]: desempacota uma lista como argumentos separados de select.
DIMENSOES = (
    ("customers", "customer", "customer_id",
     ["customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix"]),
    ("sellers", "seller", "seller_id",
     ["seller_city", "seller_state", "seller_zip_code_prefix"]),
)


def registrar_dim_versao(entidade: str, singular: str, chave: str, atributos: list) -> None:
    @dp.materialized_view(
        name=f"{CATALOG}.gold.dim_{singular}",
        comment=(
            f"Dimensão de {singular} por versão do cadastro (SCD2). {singular}_sk identifica a versão; "
            f"valido_de/valido_ate dão o intervalo (valido_ate nulo = vigente). Fonte: silver.{entidade}_historico."
        ),
    )
    def _dim():
        h = silver(f"{entidade}_historico")
        mesmo_cadastro = Window.partitionBy(chave)
        return h.select(
            F.sha2(F.concat_ws("|", F.col(chave), F.col("__START_AT").cast("string")), 256).alias(f"{singular}_sk"),
            F.col(chave),
            *[F.col(a) for a in atributos],
            F.col("__START_AT").alias("valido_de"),
            F.col("__END_AT").alias("valido_ate"),
            F.col("__END_AT").isNull().alias("versao_vigente"),
            (F.max(F.col("__END_AT").isNull().cast("int")).over(mesmo_cadastro) == 0)
            .alias("cadastro_excluido_hoje"),
        )


for entidade, singular, chave, atributos in DIMENSOES:
    registrar_dim_versao(entidade, singular, chave, atributos)

## `fact_orders`: um pedido, uma linha

O que a silver não tem como coluna é reconstruído aqui a partir dos eventos (G2):

| Coluna | Como é derivada |
|---|---|
| `status_atual` | `status` do evento de **maior `event_sequence`**; pedido sem evento → `desconhecido` |
| `order_delivered_customer_date` | `event_ts` do evento `delivered` (no máximo um por pedido) |
| `pedido_valido` | tem ao menos 1 item **e** `status_atual` não é `canceled` nem `unavailable` (ADR 0002) |
| `entregue` | `status_atual = 'delivered'` e data de entrega não nula |
| `entregue_no_prazo` | só para pedidos entregues: `DATE(entrega) <= DATE(estimada)`. **Nulo** nos demais, para que ninguém use `false` como denominador por engano |
| `lead_time_dias` | dias (com fração) entre a compra e a entrega, só para entregues |
| `atraso_dias` | `DATE(entrega) - DATE(estimada)`; positivo = atrasou. Só para entregues |
| `review_id` / `review_score` | a review de maior `review_answer_timestamp`, desempate por `review_creation_date` e `review_id` |
| `customer_sk` | versão do cliente vigente em `order_purchase_timestamp` (`>= valido_de` e `< valido_ate`) |
| `pedidos_validos_do_cliente` | (etapa 4.2) quantos pedidos válidos o `customer_unique_id` tem **no total**, em qualquer período; base da taxa de recompra |

Os timestamps não são convertidos de fuso: a silver já está no relógio local do Brasil.

In [0]:
# O QUE FAZ: declara a materialized view fact_orders, uma linha por pedido, juntando o pedido ao
#   status atual, à data de entrega, aos totais dos itens, à review mais recente e à versão do
#   cliente na data da compra. Calcula as flags e as medidas de entrega.
#
# FUNÇÕES DECLARADAS:
# - fact_orders(): devolve o DataFrame da materialized view. cluster_by pede liquid clustering por
#   data da compra e cliente único (ganho educativo com ~100 mil linhas; ajuste fino na Fase 8).
#
# FUNÇÕES NATIVAS:
# - F.row_number().over(Window.partitionBy("order_id").orderBy(...)): numera as linhas de cada pedido
#   na ordem pedida; rn = 1 é a "escolhida". .desc() = decrescente (nulos por último).
# - DataFrame.where("rn = 1"): filtra linhas (aceita expressão SQL em texto).
# - DataFrame.groupBy("order_id").agg(...): agrega por pedido. F.count("*") conta linhas, F.sum soma,
#   F.countDistinct conta valores diferentes, F.min pega o menor.
# - F.when(cond, valor): condicional; sem .otherwise o resultado é nulo quando a condição é falsa.
# - F.coalesce(a, b): primeiro valor não nulo (usado para 0 itens e para status desconhecido).
# - F.col("x").isin(a, b): verdadeiro se x é um dos valores. ~expressão nega.
# - F.sum(valido.cast("int")).over(Window.partitionBy("customer_unique_id")): soma, para cada pedido,
#   quantos pedidos válidos o mesmo cliente único tem em todo o histórico (etapa 4.2, taxa de recompra).
# - F.to_date(timestamp): corta a hora. F.datediff(fim, início): dias entre duas datas.
# - F.unix_timestamp(ts): segundos desde 1970; a diferença / 86400 dá dias com fração.
# - F.date_format(data, "yyyyMMdd").cast("int"): date_key, a chave de dim_date.
# - join com condição de intervalo: a versão do cliente vale se compra >= valido_de E (compra <
#   valido_ate OU valido_ate é nulo). É a junção por intervalo do ADR 0006, feita aqui uma vez.
@dp.materialized_view(
    name=f"{CATALOG}.gold.fact_orders",
    comment=(
        "Fato de pedidos: uma linha por pedido, com status atual e data de entrega derivados dos eventos, "
        "flags (pedido_valido, entregue, entregue_no_prazo), GMV, lead time, review mais recente e a "
        "versão do cliente vigente na compra (customer_sk)."
    ),
    cluster_by=["order_purchase_date", "customer_unique_id"],
)
def fact_orders():
    o = silver("orders").alias("o")
    eventos = silver("order_status_events")

    por_sequencia = Window.partitionBy("order_id").orderBy(F.col("event_sequence").desc())
    ultimo = (
        eventos.withColumn("rn", F.row_number().over(por_sequencia))
        .where("rn = 1")
        .select("order_id", F.col("status").alias("status_evento"))
    ).alias("u")

    entrega = (
        eventos.where(F.col("status") == "delivered")
        .groupBy("order_id")
        .agg(F.min("event_ts").alias("order_delivered_customer_date"))
    ).alias("e")

    itens = (
        silver("order_items")
        .groupBy("order_id")
        .agg(
            F.count("*").alias("qtd_itens"),
            F.countDistinct("seller_id").alias("qtd_vendedores"),
            F.sum("price").alias("valor_itens"),
            F.sum("freight_value").alias("valor_frete"),
        )
    ).alias("i")

    mais_recente = Window.partitionBy("order_id").orderBy(
        F.col("review_answer_timestamp").desc(),
        F.col("review_creation_date").desc(),
        F.col("review_id").desc(),
    )
    review = (
        silver("order_reviews")
        .withColumn("rn", F.row_number().over(mais_recente))
        .where("rn = 1")
        .select("order_id", "review_id", "review_score")
    ).alias("r")

    c = gold("dim_customer").select(
        "customer_id", "customer_sk", "customer_unique_id", "valido_de", "valido_ate",
        F.col("cadastro_excluido_hoje").alias("cliente_excluido_hoje"),
    ).alias("c")

    compra = F.col("o.order_purchase_timestamp")
    junta = (
        o.join(ultimo, "order_id", "left")
        .join(entrega, "order_id", "left")
        .join(itens, "order_id", "left")
        .join(review, "order_id", "left")
        .join(
            c,
            (F.col("o.customer_id") == F.col("c.customer_id"))
            & (compra >= F.col("c.valido_de"))
            & ((compra < F.col("c.valido_ate")) | F.col("c.valido_ate").isNull()),
            "left",
        )
    )

    status = F.coalesce(F.col("status_evento"), F.lit("desconhecido"))
    entregue = (status == "delivered") & F.col("order_delivered_customer_date").isNotNull()
    valido = (F.coalesce(F.col("qtd_itens"), F.lit(0)) > 0) & ~status.isin("canceled", "unavailable")
    pedidos_do_cliente = Window.partitionBy(F.coalesce(F.col("c.customer_unique_id"), F.col("o.customer_id")))
    qtd_itens = F.coalesce(F.col("qtd_itens"), F.lit(0))
    valor_itens = F.coalesce(F.col("valor_itens"), F.lit(0))
    valor_frete = F.coalesce(F.col("valor_frete"), F.lit(0))
    estimada = F.col("o.order_estimated_delivery_date")
    entrega_ts = F.col("order_delivered_customer_date")

    return junta.select(
        F.col("o.order_id").alias("order_id"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("c.customer_sk").alias("customer_sk"),
        F.col("c.customer_unique_id").alias("customer_unique_id"),
        F.col("cliente_excluido_hoje"),
        compra.alias("order_purchase_timestamp"),
        F.to_date(compra).alias("order_purchase_date"),
        F.date_format(compra, "yyyyMMdd").cast("int").alias("date_key"),
        estimada.alias("order_estimated_delivery_date"),
        entrega_ts,
        status.alias("status_atual"),
        qtd_itens.alias("qtd_itens"),
        F.coalesce(F.col("qtd_vendedores"), F.lit(0)).alias("qtd_vendedores"),
        valor_itens.alias("valor_itens"),
        valor_frete.alias("valor_frete"),
        (valor_itens + valor_frete).alias("gmv_bruto"),
        (qtd_itens > 0).alias("tem_item"),
        valido.alias("pedido_valido"),
        F.sum(valido.cast("int")).over(pedidos_do_cliente).alias("pedidos_validos_do_cliente"),
        (status == "canceled").alias("cancelado"),
        (status == "unavailable").alias("indisponivel"),
        entregue.alias("entregue"),
        F.when(entregue, F.to_date(entrega_ts) <= F.to_date(estimada)).alias("entregue_no_prazo"),
        F.when(entregue, (F.unix_timestamp(entrega_ts) - F.unix_timestamp(compra)) / 86400)
        .alias("lead_time_dias"),
        F.when(entregue, F.datediff(F.to_date(entrega_ts), F.to_date(estimada))).alias("atraso_dias"),
        F.col("r.review_id").alias("review_id"),
        F.col("r.review_score").alias("review_score"),
    )

## `fact_order_items` e `fact_reviews`

**`fact_order_items`** (uma linha por item) herda do pedido a data da compra, o status e a flag `pedido_valido`, e ganha a versão do **vendedor** vigente na compra (`seller_sk`). Assim o GMV de um recorte é só `SUM(item_total) WHERE pedido_valido`, sem juntar mais nada. Lê `fact_orders` do próprio pipeline.

**`fact_reviews`** (uma linha por par review × pedido: o `review_id` se repete em 795 linhas, mas o par é único) marca a review que vale por pedido em `eh_mais_recente` e as reviews cujo pedido não existe em `pedido_existe = false` (17 casos, fora das métricas). O texto do comentário **não** vai para a gold: só `tem_comentario`. Texto livre pode conter dado pessoal, e a Fase 6 trata isso na silver.

In [0]:
# O QUE FAZ: declara fact_order_items (itens com a data/status do pedido e a versão do vendedor na
#   compra) e fact_reviews (reviews com a marca da mais recente por pedido).
#
# FUNÇÕES DECLARADAS:
# - fact_order_items(): lê itens da silver e fact_orders/dim_seller da gold; junção por intervalo
#   para achar o seller_sk.
# - fact_reviews(): lê reviews e pedidos da silver.
#
# FUNÇÕES NATIVAS:
# - DataFrame.join(outro, "order_id", "left"): junção pela coluna de mesmo nome; "left" mantém todos
#   os itens/reviews mesmo sem correspondência (um item sem pedido apareceria com nulos, em vez de sumir).
# - Junção por intervalo: compra >= valido_de E (compra < valido_ate OU valido_ate nulo).
# - F.col("a") + F.col("b"): soma de colunas DECIMAL (preço + frete) sem perder precisão.
# - F.row_number().over(Window.partitionBy("order_id").orderBy(...)): marca a review que vale
#   (rn = 1) dentro de cada pedido; o desempate é o mesmo da fact_orders.
# - F.col("pedido_existe") & (rn == 1): "e" lógico entre colunas booleanas.
# - F.length(F.trim(coluna)): tamanho do texto sem espaços nas pontas; > 0 = tem texto.
@dp.materialized_view(
    name=f"{CATALOG}.gold.fact_order_items",
    comment=(
        "Fato de itens: uma linha por item de pedido, com preço, frete, item_total, status e pedido_valido do "
        "pedido e a versão do vendedor vigente na compra (seller_sk). GMV = soma de item_total onde pedido_valido."
    ),
    cluster_by=["order_purchase_date", "seller_id"],
)
def fact_order_items():
    i = silver("order_items").alias("i")
    o = gold("fact_orders").select(
        "order_id", "order_purchase_timestamp", "order_purchase_date", "date_key",
        "status_atual", "pedido_valido",
    ).alias("o")
    s = gold("dim_seller").select(
        "seller_id", "seller_sk", "valido_de", "valido_ate",
        F.col("cadastro_excluido_hoje").alias("vendedor_excluido_hoje"),
    ).alias("s")

    compra = F.col("o.order_purchase_timestamp")
    junta = i.join(o, F.col("i.order_id") == F.col("o.order_id"), "left").join(
        s,
        (F.col("i.seller_id") == F.col("s.seller_id"))
        & (compra >= F.col("s.valido_de"))
        & ((compra < F.col("s.valido_ate")) | F.col("s.valido_ate").isNull()),
        "left",
    )
    return junta.select(
        F.col("i.order_id").alias("order_id"),
        F.col("i.order_item_id").alias("order_item_id"),
        F.col("i.product_id").alias("product_id"),
        F.col("i.seller_id").alias("seller_id"),
        F.col("s.seller_sk").alias("seller_sk"),
        F.col("vendedor_excluido_hoje"),
        compra.alias("order_purchase_timestamp"),
        F.col("o.order_purchase_date").alias("order_purchase_date"),
        F.col("o.date_key").alias("date_key"),
        F.col("i.shipping_limit_date").alias("shipping_limit_date"),
        F.col("i.price").alias("price"),
        F.col("i.freight_value").alias("freight_value"),
        (F.col("i.price") + F.col("i.freight_value")).alias("item_total"),
        F.col("o.status_atual").alias("status_atual"),
        F.col("o.pedido_valido").alias("pedido_valido"),
    )


@dp.materialized_view(
    name=f"{CATALOG}.gold.fact_reviews",
    comment=(
        "Fato de reviews: uma linha por par (review, pedido). eh_mais_recente marca a review que vale por pedido; "
        "pedido_existe = false são reviews sem pedido (fora das métricas). Sem o texto do comentário."
    ),
)
def fact_reviews():
    r = silver("order_reviews").alias("r")
    pedidos = silver("orders").select(F.col("order_id").alias("pedido_order_id")).alias("p")

    mais_recente = Window.partitionBy("r.order_id").orderBy(
        F.col("r.review_answer_timestamp").desc(),
        F.col("r.review_creation_date").desc(),
        F.col("r.review_id").desc(),
    )
    junta = r.join(pedidos, F.col("r.order_id") == F.col("p.pedido_order_id"), "left")
    existe = F.col("p.pedido_order_id").isNotNull()
    return junta.select(
        F.col("r.review_id").alias("review_id"),
        F.col("r.order_id").alias("order_id"),
        existe.alias("pedido_existe"),
        (existe & (F.row_number().over(mais_recente) == 1)).alias("eh_mais_recente"),
        F.col("r.review_score").alias("review_score"),
        F.col("r.review_creation_date").alias("review_creation_date"),
        F.col("r.review_answer_timestamp").alias("review_answer_timestamp"),
        (F.length(F.trim(F.col("r.review_comment_message"))) > 0).alias("tem_comentario"),
    )

## `gold_seller_mensal` (etapa 4.2)

Base da métrica **concentração de vendas**: uma linha por vendedor e mês da compra, só com pedidos válidos. `rank_no_mes` ordena os vendedores do mês pelo GMV (do maior para o menor; empate decidido por `seller_id`, para o resultado ser sempre o mesmo). A Metric View de concentração soma o GMV dos que têm `rank_no_mes <= 10`. É um agregado porque "o GMV dos 10 maiores" é uma agregação sobre outra agregação, que uma medida simples não expressa.

In [0]:
# O QUE FAZ: declara a materialized view gold_seller_mensal: GMV por vendedor e mês da compra, com a
#   posição do vendedor no mês (rank_no_mes), usando só itens de pedidos válidos.
#
# FUNÇÕES DECLARADAS:
# - gold_seller_mensal(): lê gold.fact_order_items e agrega.
#
# FUNÇÕES NATIVAS:
# - DataFrame.where("pedido_valido"): mantém só itens de pedidos válidos (a coluna já é booleana).
# - F.date_format(data, "yyyy-MM"): mês da compra como texto.
# - groupBy(...).agg(F.sum, F.countDistinct): GMV (preço + frete) e número de pedidos do vendedor no mês.
# - F.row_number().over(Window.partitionBy("ano_mes").orderBy(F.col("gmv").desc(), "seller_id")):
#   posição do vendedor dentro do mês; seller_id desempata, para não haver duas posições iguais.
@dp.materialized_view(
    name=f"{CATALOG}.gold.gold_seller_mensal",
    comment=(
        "GMV por vendedor e mês da compra (só pedidos válidos), com rank_no_mes. Base da métrica de "
        "concentração de vendas (top 10 do mês)."
    ),
)
def gold_seller_mensal():
    itens = (
        gold("fact_order_items")
        .where("pedido_valido")
        .groupBy(F.date_format("order_purchase_date", "yyyy-MM").alias("ano_mes"), "seller_id")
        .agg(F.sum("item_total").alias("gmv"), F.countDistinct("order_id").alias("pedidos"))
    )
    posicao = Window.partitionBy("ano_mes").orderBy(F.col("gmv").desc(), F.col("seller_id"))
    return itens.withColumn("rank_no_mes", F.row_number().over(posicao))

## Como executar e conferir

Este bloco é só roteiro, não é código do pipeline.

**1. Criar o pipeline `olist.gold`** (G1): *Jobs & Pipelines → Create → ETL pipeline*.
- Nome `olist.gold` · catálogo `olist` · schema de destino `gold` · serverless.
- *Source code*: este arquivo (`src/gold/01_gold_modelo`).
- *Configuration*: `olist.catalogo` = `olist` (como no silver; sem a chave ele já usa `olist`).
- Se o Free Edition **recusar** o segundo pipeline, é o plano reserva: *Settings → Source code* do `olist.silver` e **adicione** este arquivo. Os nomes completos (`olist.gold.<tabela>`) permitem isso.

**2. *Dry run*** primeiro: o grafo deve mostrar 8 tabelas (a `gold_seller_mensal` entrou na etapa 4.2), com `fact_orders` ligada a `dim_customer` e `fact_order_items` ligada a `fact_orders` e `dim_seller`. Erro de `cluster_by` ou de nome aparece aqui. **3. *Run pipeline*** normal.

**4. Conferir** no SQL editor (esperado entre comentários):

```sql
-- granularidade
SELECT 'dim_date' t, count(*) n FROM olist.gold.dim_date UNION ALL
SELECT 'dim_product', count(*) FROM olist.gold.dim_product UNION ALL     -- 32.951
SELECT 'dim_customer', count(*) FROM olist.gold.dim_customer UNION ALL   -- 101.008
SELECT 'dim_seller', count(*) FROM olist.gold.dim_seller UNION ALL       -- 3.361
SELECT 'fact_orders', count(*) FROM olist.gold.fact_orders UNION ALL     -- 98.476
SELECT 'fact_order_items', count(*) FROM olist.gold.fact_order_items UNION ALL  -- 111.592
SELECT 'fact_reviews', count(*) FROM olist.gold.fact_reviews;            -- 95.253

-- chave substituta: nenhum fato sem dimensão (esperado 0 em todas)
SELECT count_if(customer_sk IS NULL) AS pedidos_sem_cliente,
       count(*) - count(DISTINCT order_id) AS pedidos_duplicados
FROM olist.gold.fact_orders;
SELECT count_if(seller_sk IS NULL) AS itens_sem_vendedor,
       count(*) - count(DISTINCT order_id, order_item_id) AS itens_duplicados
FROM olist.gold.fact_order_items;
SELECT count(*) FROM olist.gold.fact_orders f LEFT ANTI JOIN olist.gold.dim_date d USING (date_key);   -- 0
SELECT count(*) FROM olist.gold.fact_order_items f LEFT ANTI JOIN olist.gold.dim_product p USING (product_id);  -- 0
SELECT count(*) - count(DISTINCT customer_sk) FROM olist.gold.dim_customer;   -- 0 (chave única)
SELECT count(*) - count(DISTINCT seller_sk) FROM olist.gold.dim_seller;       -- 0

-- status atual (esperado: o mesmo da silver)
SELECT status_atual, count(*) FROM olist.gold.fact_orders GROUP BY 1 ORDER BY 2 DESC;
-- delivered 93.247 · shipped 2.628 · approved 657 · unavailable 608 · canceled 584 · invoiced 314 · processing 301 · created 135 · desconhecido 2

-- totais
SELECT sum(item_total) AS gmv_bruto_itens FROM olist.gold.fact_order_items;     -- 15.722.601,92
SELECT sum(gmv_bruto) AS gmv_bruto_pedidos FROM olist.gold.fact_orders;         -- 15.722.601,92
SELECT count_if(pedido_valido) AS validos, count_if(tem_item) AS com_item,
       count_if(entregue) AS entregues, count_if(entregue_no_prazo) AS no_prazo,
       count_if(lead_time_dias < 0) AS lead_negativo
FROM olist.gold.fact_orders;                                                    -- lead_negativo = 0

-- reviews: uma que vale por pedido, 17 sem pedido
SELECT count_if(eh_mais_recente) AS validas, count(DISTINCT CASE WHEN pedido_existe THEN order_id END) AS pedidos_com_review,
       count_if(NOT pedido_existe) AS orfas
FROM olist.gold.fact_reviews;                                                   -- validas = pedidos_com_review · orfas 17
```

**Segunda execução** (*Run pipeline* de novo, sem dado novo): as contagens não podem mudar e o pipeline deve atualizar de forma incremental ou pular as views (olhe a coluna de *Rows written* no grafo).

**Inversões de `event_ts`:** 1.370 pedidos têm algum evento com `event_ts` anterior a um evento de sequência menor (127 deles no evento final); só 223 têm falha registrada em `ops.simulator_chaos_log`. A maior parte vem do dataset original, e é por isso que o status usa `event_sequence`, não `event_ts`.